In [1]:
!pip install easyocr sentence-transformers faiss-cpu pymupdf pillow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 29.1 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()


Saving resume.pdf to resume.pdf


In [3]:
import fitz  # PyMuPDF
from PIL import Image

pdf_path = "resume.pdf"
doc = fitz.open(pdf_path)

image_paths = []

for i, page in enumerate(doc):
    pix = page.get_pixmap(dpi=300)
    img_path = f"page_{i}.png"
    pix.save(img_path)
    image_paths.append(img_path)

print("Total pages:", len(image_paths))


Total pages: 4


In [4]:
import easyocr

reader = easyocr.Reader(['en'])
extracted_text = ""

for img in image_paths:
    results = reader.readtext(img)
    for res in results:
        extracted_text += res[1] + " "

print(extracted_text[:1000])  # first 1000 chars


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteMst Moushumi Khatun Passport: A15479811 Date of birth: 25/12/1999 Place of birth: Dhaka, Bangladesh Nationality: Bangladeshi Phone number: (+880) 01946615954 (Home) Email address: moushumipriya1999@gmailcom Website: https: ILmoushumipriya githubioL Linkedln: bttpsILwwwlinkedin com/in/mst-moushumi-kL GitHub https:ILgithubcom/moushumipriya Whatsapp Messenger: 01946615954 Address: Rani nagar, 6760, Boalia bazar, Bangladesh (Home) ABOUT ME Al Engineer with expertise in Machine Learning, Deep Learning, NLP, and Data Science, specializing in Europe- relevant applications across maritime & fisheries Al, healthcare, autonomous systems, and multilingual legal automation: Experienced in building end-to-end Al solutions-_from data preprocessing and model development to deployment in production environments. Published in ACM and Springer, with a strong portfolio of real-world projects targeting sustainable technology, he

In [5]:
def chunk_text(text, chunk_size=150):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i+chunk_size]))
    return chunks

chunks = chunk_text(extracted_text)
print("Total chunks:", len(chunks))
print(chunks[0])


Total chunks: 7
Mst Moushumi Khatun Passport: A15479811 Date of birth: 25/12/1999 Place of birth: Dhaka, Bangladesh Nationality: Bangladeshi Phone number: (+880) 01946615954 (Home) Email address: moushumipriya1999@gmailcom Website: https: ILmoushumipriya githubioL Linkedln: bttpsILwwwlinkedin com/in/mst-moushumi-kL GitHub https:ILgithubcom/moushumipriya Whatsapp Messenger: 01946615954 Address: Rani nagar, 6760, Boalia bazar, Bangladesh (Home) ABOUT ME Al Engineer with expertise in Machine Learning, Deep Learning, NLP, and Data Science, specializing in Europe- relevant applications across maritime & fisheries Al, healthcare, autonomous systems, and multilingual legal automation: Experienced in building end-to-end Al solutions-_from data preprocessing and model development to deployment in production environments. Published in ACM and Springer, with a strong portfolio of real-world projects targeting sustainable technology, healthcare innovation, and public service automation: WORK EXPER

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(chunks)
embeddings = np.array(embeddings).astype("float32")

print("Embedding shape:", embeddings.shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (7, 384)


In [7]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Total vectors in index:", index.ntotal)


Total vectors in index: 7


In [8]:
query = "What skills does the candidate have?"

query_embedding = model.encode([query]).astype("float32")

D, I = index.search(query_embedding, k=1)

print("✅ ANSWER:\n")
print(chunks[I[0][0]])


✅ ANSWER:

Machine Intelligence Research Labblished in ACM and Springer. BRITSYNC WALES UNITED KINGDOM SENIOR INSTRUCTOR-REMOTE 01/08/2025 CURRENT Designed and developed Al and Data Science courses for students and professionals Mentored junior team members and guided learners to achieve practical competency in AI/ML Created curriculum outlines, coding exercises, and project-based assessments aligned with international Al standards_. BUSINESS AUTOMATION LTD DHAKA, BANGLADESH TRAINEE SOFTWARE DEVELOPER 20/07/2024 _ 04/09/2025 Developed and maintained web modules using Laravel (PHP), HTML, CSS, and JavaScript; Participated in SQA processes including test case creation and reporting: Worked in Agile teams and managed code via Git; BRITSYNC ~ WALES UNITED KINGDOM JUNIOR DATA SCIENTIST-REMOTE 01/07/2024 31/07/2025 Developed and optimized machine learning models for Al projects.Conducted EDA to extract actionable insights, improving decision-making efficiency: Performed data preprocessing, c

In [9]:
query = "What programming languages are mentioned?"

query_embedding = model.encode([query]).astype("float32")

D, I = index.search(query_embedding, k=1)

print("✅ ANSWER:\n")
print(chunks[I[0][0]])


✅ ANSWER:

Collected, cleaned, and organized 5,000+ image datasets for computer vision model training: Performed image preprocessing (resizing, normalization, augmentation), improving model accuracy by 20%. bug Annotated images using Labellmg, CVAT, Roboflow, ensuring high-quality bounding boxes and segmentation masks. Collaborated with the Al team to build training datasets for object detection and classification models (YOLO, CNN): SKILLS Languages: Python, PHP, JavaScript, SQL, HTML, CSS MLIDL Frameworks: TensorFloW, PyTorch, Keras, scikit-learn, XGBoost; LSTM, CNN NLP & RAG: LangChain, Hugging Face Transformers, spaCy, NLTK, Translation Models Data Tools: Pandas, NumPy, Matplotlib, Seaborn, Power Bl, Excel Cloud & DevOps: AWS (EC2), Docker, Kubernetes, Git, Flask; Streamlit Other: Agile/Scrum, SQA, Manual Testing, REST API development PROJECTS 06/06/2025 07/06/2025 Multilingual Norwegian Constitution Chatbot (NLP + RAG) Chatbot answering queries from Norwegian legal documents in Be

In [11]:
"What is the candidate's name?"
"What is the education background?"
"What programming languages are mentioned?"
"What experience does the candidate have?"


'What experience does the candidate have?'